# Premarket / Extended-Hours SMART Stock Bars Example

This example subscribes with `useRTH=False` on `SMART` and allows orders outside regular trading hours with `outside_rth=True`.

In [1]:
from __future__ import annotations

import copy
import sys
from pathlib import Path

from ib_async import IB, util

# Required for sync ib.connect(...) inside Jupyter/IPython kernels.
util.startLoop()

repo_root = Path.cwd().resolve()
if repo_root.name == "examples":
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from config import DEFAULT_LIVE_CONFIG, LiveTradingConfig
from execution import (
    active_orders,
    filled_orders,
    recent_fills,
    strategy_open_trades,
    print_order_snapshot,
    build_execution_instrument,
    request_underlying_realtime_bars,
)
from orders import IBKRLimitOrderRouter
from strategies.live_mean_reversion import LiveMeanReversion


In [2]:
SYMBOL = "META"
IB_HOST = "127.0.0.1"
IB_PORT = 4002
IB_CLIENT_ID = 111

raw = copy.deepcopy(DEFAULT_LIVE_CONFIG)
raw["symbol"] = SYMBOL
raw["model"]["enabled"] = False
raw["features"]["window"] = 2
raw["features"]["min_bars"] = 3
raw["strategy"]["entry_strategy"] = {"name": "always_true"}
raw["strategy"]["price_only_entries"] = True
raw["strategy"]["require_rsi"] = False
raw["strategy"]["no_trade_first_minutes"] = -1000
raw["strategy"]["no_new_entries_last_minutes"] = -1000
raw["strategy"]["entry_price_limits"] = {
    "first_entry": {"max_stock_price": 750.0, "max_buy_price": 750.0},
}
raw["double_down"]["price_rules"] = []
raw["strategy"]["latest_trade_price_rules"] = {
    "enabled": True,
    "first_entry": [],
    "double_down": [
        {"basis": "execution", "mode": "abs", "max_change": -5.00},
    ],
}
raw["execution"]["market_data"] = {
    "exchange": "SMART",
    "barSize": 5,
    "whatToShow": "TRADES",
    "useRTH": False,
}
raw["execution"]["outside_rth"] = True
raw["execution"]["instrument"] = {
    "type": "stock",
    "exchange": "SMART",
    "currency": "USD",
    "limit_entry_offset_pct": 0.0005,
    "limit_exit_offset_pct": 0.0005,
}

config = LiveTradingConfig.from_dict(raw)
config.execution, config.strategy["latest_trade_price_rules"]


({'tif': 'DAY',
  'outside_rth': True,
  'market_data': {'exchange': 'SMART',
   'barSize': 5,
   'whatToShow': 'TRADES',
   'useRTH': False},
  'instrument': {'type': 'stock',
   'exchange': 'SMART',
   'currency': 'USD',
   'limit_entry_offset_pct': 0.0005,
   'limit_exit_offset_pct': 0.0005}},
 {'enabled': True,
  'first_entry': [],
  'double_down': [{'basis': 'execution', 'mode': 'abs', 'max_change': -5.0}]})

In [3]:
ib = IB()
if not ib.isConnected():
    ib.connect(IB_HOST, IB_PORT, clientId=IB_CLIENT_ID)

stock, real_time_bars = request_underlying_realtime_bars(
    ib=ib,
    symbol=config.symbol,
    market_data_cfg=config.execution["market_data"],
)

execution_instrument = build_execution_instrument(ib, config.execution, config.symbol)
order_router = IBKRLimitOrderRouter(ib=ib, contract=stock)
algo = LiveMeanReversion(config=config, order_router=order_router, execution_instrument=execution_instrument)

print("Market data contract:", stock)
print("useRTH:", config.execution["market_data"]["useRTH"])
print("outside_rth orders:", config.execution["outside_rth"])


[LOAD OPEN TRADES] empty/corrupt file ignored: META_open_trades.csv
Market data contract: Stock(conId=107113386, symbol='META', exchange='SMART', primaryExchange='NASDAQ', currency='USD', localSymbol='META', tradingClass='NMS')
useRTH: False
outside_rth orders: True
[BOOTSTRAP] loaded 0 completed signal bars | timeframe=1min
[NEW SIGNAL BAR] timeframe=1min time=2026-06-01 13:23:00+00:00 O=632.0 H=632.01 L=631.73 C=631.98 V=1478.0 | stored=1
[NEW SIGNAL BAR] timeframe=1min time=2026-06-01 13:24:00+00:00 O=631.98 H=631.98 L=631.49 C=631.49 V=1259.0 | stored=2
[NEW SIGNAL BAR] timeframe=1min time=2026-06-01 13:25:00+00:00 O=631.0 H=631.21 L=629.8 C=630.5 V=6983.0 | stored=3
[MODEL] 2026-06-01 13:25:00+00:00 enabled=False prob=1.0000 passed=True missing=[]
[FEATURES] 2026-06-01 13:25:00+00:00 close=630.50 median=631.00 mad=0.3700 bb_score=-1.34 rsi=None ml_prob=1.0000 session_min=235 mins_to_close=154
[NEW DAY] 2026-06-01
[SAVE OPEN TRADES] saved 0 rows to META_open_trades.csv


Exception in callback _SelectorSocketTransport._read_ready()
handle: <Handle _SelectorSocketTransport._read_ready()>
Traceback (most recent call last):
  File "/opt/miniconda3/lib/python3.13/asyncio/events.py", line 89, in _run
    self._context.run(self._callback, *self._args)
    ~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
RuntimeError: cannot enter context: <_contextvars.Context object at 0x1049613c0> is already entered
Exception in callback _SelectorSocketTransport._read_ready()
handle: <Handle _SelectorSocketTransport._read_ready()>
Traceback (most recent call last):
  File "/opt/miniconda3/lib/python3.13/asyncio/events.py", line 89, in _run
    self._context.run(self._callback, *self._args)
    ~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
RuntimeError: cannot enter context: <_contextvars.Context object at 0x1049613c0> is already entered
Exception in callback _SelectorSocketTransport._read_ready()
handle: <Handle _SelectorSocketTransport._read_ready()>
Traceback (most recent 

In [4]:
# Attach when ready.
real_time_bars.updateEvent += algo.on_bar

# Detach before rerunning setup.
# real_time_bars.updateEvent -= algo.on_bar


## Status Checks

Use these cells to see whether the strategy is waiting for bars, blocked by config, has active orders, or has fills.


In [18]:
print_order_snapshot(ib, algo)
print("completed_signal_bars:", len(algo.signal_bars))
print("current_partial_bar:", algo.signal_bar_builder.current_bar)
print("last_signal:", algo.last_signal)


Order snapshot: 2026-06-01T16:25:43
Active IB orders: 0
Filled IB orders: 0
Recent fills: 0
Strategy open trades: 0
completed_signal_bars: 2
current_partial_bar: {'time': datetime.datetime(2026, 6, 1, 13, 25, tzinfo=datetime.timezone.utc), 'open': 631.0, 'high': 631.21, 'low': 629.8, 'close': 629.8, 'volume': 4493.0}
last_signal: None


In [19]:
active_orders(ib)


""


In [20]:
filled_orders(ib)


""


In [21]:
recent_fills(ib)


""


In [22]:
strategy_open_trades(algo)


""


In [23]:
features = algo.calculate_features()
if features is None:
    print("features not ready yet; need more completed signal bars")
else:
    signal = algo.should_enter_trade(features)
    print("candidate_signal:", signal)


features not ready yet; need more completed signal bars
